In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# from tqdm import tqdm
from glob import glob
from typing import Any, TypedDict
from matplotlib.axes import Axes
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from typing import List, Dict, Tuple, Any, Optional
from matplotlib.ticker import AutoMinorLocator

def set_grid(
    ax: Axes,
    n_locator: int = 10,
    minor_line_width: float = 0.2,
    major_line_width: float = 0.4,
) -> None:
    ax.xaxis.set_minor_locator(AutoMinorLocator(n_locator))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n_locator))
    ax.grid(which="minor", linestyle="--", linewidth=minor_line_width)
    ax.grid(which="major", linewidth=major_line_width)


def is_rgb_histograms(data: dict) -> bool:
    return isinstance(data, dict) and set(data.keys()) == {"r", "g", "b"}


def verbose_plot(
    to_plot: Any,
    title: str = "",
    x_label: str = "",
    y_label: str = "",
    figure_size: tuple[float, float] = (10, 6),
    is_image: bool = False,
) -> None:
    if to_plot is None or (isinstance(to_plot, np.ndarray) and to_plot.size == 0):
        return

    _, axs = plt.subplots(1, 1, figsize=figure_size)

    axs.set_title(title)
    axs.set_xlabel(x_label)
    axs.set_ylabel(y_label)

    if isinstance(to_plot, dict):
        if is_rgb_histograms(to_plot):
            for color in ("r", "g", "b"):
                axs.plot(to_plot[color], color=color, label=color.upper())
        else:
            for key, value in to_plot.items():
                axs.plot(value, label=key)
        axs.legend()
    else:
        if is_image:
            axs.imshow(to_plot)
            axs.axis("off")
        else:
            axs.plot(to_plot)

    set_grid(axs)
    plt.show()


def puzzle_images(images_paths: list) -> None:
    n_features = 1000
    n_octave_layers = 3
    contrast_threshold = 0.04
    edge_threshold = 8
    sigma = 1.3

    detector = cv2.SIFT_create(
        nfeatures=n_features,
        nOctaveLayers=n_octave_layers,
        contrastThreshold=contrast_threshold,
        edgeThreshold=edge_threshold,
        sigma=sigma,
    )

    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    flann = cv2.BFMatcher(cv2.NORM_L2, False)
    image_base = cv2.imread(images_paths[0])

    num_images = len(images_paths)
    grid_size = int(np.sqrt(num_images))
    H, W = (image_base.shape[0] * 2 * grid_size,
            image_base.shape[1] * 2 * grid_size)

    final = np.zeros((H, W, 3))
    row_start = H // 2 - image_base.shape[0] // 2
    row_end = H // 2 + image_base.shape[0] // 2
    col_start = W // 2 - image_base.shape[1] // 2
    col_end = W // 2 + image_base.shape[1] // 2
    final[row_start:row_end, col_start:col_end] = image_base / 255
    del image_base

    bases = [[] for _ in range(num_images)]
    candidates = list(range(1, num_images))
    center = [0]
    checked = []
    ratio_thresh = 0.47

    while center:
        i = center.pop()
        loc_image_i = cv2.imread(images_paths[i])

        yrcb_i = cv2.cvtColor(loc_image_i, cv2.COLOR_RGB2YCrCb)
        bright_i = clahe.apply(yrcb_i[:, :, 0])
        yrcb_i[:, :, 0] = bright_i
        loc_image_i = cv2.cvtColor(yrcb_i, cv2.COLOR_YCrCb2RGB)

        keypoints_i, descriptors_i = detector.detectAndCompute(
            loc_image_i, None)

        if descriptors_i is None:
            continue

        while candidates:
            j = candidates.pop()
            checked.append(i)

            loc_image_j = cv2.imread(images_paths[j])

            yrcb_j = cv2.cvtColor(loc_image_j, cv2.COLOR_RGB2YCrCb)
            bright_j = clahe.apply(yrcb_j[:, :, 0])
            yrcb_j[:, :, 0] = bright_j
            loc_image_j = cv2.cvtColor(yrcb_j, cv2.COLOR_YCrCb2RGB)

            keypoints_j, descriptors_j = detector.detectAndCompute(
                loc_image_j, None)

            if descriptors_j is None:
                continue

            raw_matches = flann.knnMatch(
                np.asarray(descriptors_i, np.float32),
                np.asarray(descriptors_j, np.float32),
                k=2,
            )

            good_matches = [
                m
                for m, n in raw_matches
                if len(raw_matches[0]) >= 2 and m.distance < ratio_thresh * n.distance
            ]

            if len(good_matches) >= 3:
                points1 = np.float32(
                    [keypoints_i[match.queryIdx].pt for match in good_matches]
                )
                points2 = np.float32(
                    [keypoints_j[match.trainIdx].pt for match in good_matches]
                )

                mask = np.uint8(
                    cv2.cvtColor(final.astype(np.float32)
                                 * 255, cv2.COLOR_BGR2GRAY)
                    == 0
                )
                mask = cv2.erode(mask, np.ones((3, 3)), iterations=2)
                mask = cv2.dilate(mask, np.ones((3, 3)), iterations=2)

                out_affine = cv2.estimateAffine2D(points2, points1)

                if out_affine[0] is not None:
                    center.append(j)

                    transformed_image = np.zeros((H, W, 3))
                    row_start = H // 2 - loc_image_j.shape[0] // 2
                    row_end = H // 2 + loc_image_j.shape[0] // 2
                    col_start = W // 2 - loc_image_j.shape[1] // 2
                    col_end = W // 2 + loc_image_j.shape[1] // 2
                    transformed_image[row_start:row_end, col_start:col_end] = (
                        loc_image_j / 255
                    )

                    for base in bases[i]:
                        bases[j].append(base)
                        transformed_image = cv2.warpAffine(
                            transformed_image, base, (W, H)
                        )

                    bases[j].append(out_affine[0])

                    transformed_image = cv2.warpAffine(
                        transformed_image, out_affine[0], (W, H)
                    )

                    final += (
                        np.repeat(mask[..., np.newaxis], 3,
                                  axis=-1) * transformed_image
                    )

        candidates = [
            k for k in range(1, num_images) if k not in center and k not in checked
        ]

    verbose_plot(final, is_image=True)

china_shuffle = glob(f"{os.getcwd()}/puzzle/china_shuffle/*")

puzzle_images(china_shuffle)


ModuleNotFoundError: No module named 'tqdm'